In [7]:
import pandas as pd 
import json
import numpy
import sys
from pathlib import Path
import re
import networkx as nx





In [8]:
sys.path.insert(0, r"..\LLM_Open_IE\scripts\graphs")
from graph_plot import GraphPlotter

In [9]:
import re
pasta = Path.cwd()

while not (pasta / "Outputs").exists() and pasta != pasta.parent:
    pasta = pasta.parent


arquivos = [
    arquivo
    for arquivo in (pasta / "Outputs").rglob("Respostas*/*metricas_sentencas.csv")
    if not any(
        re.fullmatch(r"k_\d+", parte.name, re.IGNORECASE)
        for parte in arquivo.parents
    )
]

dataframes = []

for arquivo in arquivos:
    df_temp = pd.read_csv(arquivo)
    df_temp["pasta_respostas"] = arquivo.parent.name
    dataframes.append(df_temp)

df = pd.concat(dataframes, ignore_index=True)
df["k"] = df["k"].fillna("")
df = df[df["k"] == ""]

print(df)

        corpus        lote k                        prompt  \
0          bia  10_batches     Respostas AbstractiveOpenIE   
1          bia  10_batches     Respostas AbstractiveOpenIE   
2          bia  10_batches     Respostas AbstractiveOpenIE   
3          bia  10_batches     Respostas AbstractiveOpenIE   
4          bia  10_batches     Respostas AbstractiveOpenIE   
...        ...         ... ..                          ...   
14166  oiec_pt  10_batches     Respostas ExtrativoPTOIE-DP   
14167  oiec_pt  10_batches     Respostas ExtrativoPTOIE-DP   
14168  oiec_pt  10_batches     Respostas ExtrativoPTOIE-DP   
14169  oiec_pt  10_batches     Respostas ExtrativoPTOIE-DP   
14170  oiec_pt  10_batches     Respostas ExtrativoPTOIE-DP   

                     arquivo  sentenca  \
0             Claude_Sonnet5         0   
1             Claude_Sonnet5         1   
2             Claude_Sonnet5         2   
3             Claude_Sonnet5         3   
4             Claude_Sonnet5         4   
...

In [14]:
T = lambda s: [(t["arg1"],t["rel"],t["arg2"]) for t in json.loads(s)] if isinstance(s,str) else []
IMG = Path(r"Outputs\graphs\images\grau")

In [15]:
for (corpus,lote,prompt), g in df[df['prompt']!='gold'].groupby(["corpus",'lote','prompt']):
    gold = df[(df['corpus'] == corpus) & (df['prompt'] == 'gold')].drop_duplicates('frases').set_index('frase')['relacoes_json']
    plotter = GraphPlotter(str(IMG/corpus/lote/prompt))
    for i, (frase,gf) in enumerate(g[g['grau_max']>4].groupby('frase')):
        paineis = [('gold',T(gold.get('frase')))]
        paineis += [(f"{r.arquivo} (ciclo)",T(r.relacoes_json)) for r in gf.itertuples()]
        plotter.plot_triplas(paineis, frase, f"{i:03d}")